# LeafCheck – Step 2: evaluate on the held-out test set (Google Colab)

Run **after** `train.ipynb`. A GPU is not required. `Runtime → Run all`.

Produces in `MyDrive/leafcheck/`:
* `reports/classification_report.txt` – precision, recall, F1 per class
* `reports/confusion_matrix.png` – which classes get mixed up
* `reports/results.md` – the summary table for the README
* `samples/` – demo images for the web app, and `samples/test_split/` for the sanity test

The last cell downloads everything as **`leafcheck_outputs.zip`**. Unzip it into the repo root.

In [ ]:
import json, shutil, time
import numpy as np
import tensorflow as tf

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/leafcheck")  # mirrors the repo layout
for sub in ("model", "reports", "samples", "test_split"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Saving outputs to", OUT_DIR)

In [ ]:
CLASS_NAMES = [
    "Corn___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn___Common_rust",
    "Corn___Northern_Leaf_Blight",
    "Corn___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]  # same list and order as leafcheck/config.py

In [ ]:
THRESHOLD = 0.60  # same as leafcheck/config.py
IMG_SIZE = 224

model = tf.keras.models.load_model(OUT_DIR / "model" / "leafcheck_mobilenetv2.keras")
labels = json.loads((OUT_DIR / "model" / "labels.json").read_text())
assert labels == CLASS_NAMES

test_dir = OUT_DIR / "test_split"
expected = json.loads((test_dir / "expected.json").read_text())
filenames = sorted(expected)
y_true = np.array([labels.index(expected[f]) for f in filenames])
print(len(filenames), "test images")

## 1. Predict every test image

Same preprocessing as training and as the web app: decode → resize 224 → scale to [-1, 1].

In [ ]:
def load(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), antialias=True)
    return tf.keras.applications.mobilenet_v2.preprocess_input(image)

paths = [str(test_dir / f) for f in filenames]
test_ds = tf.data.Dataset.from_tensor_slices(paths).map(load).batch(64)

start = time.perf_counter()
probs = model.predict(test_ds)
per_image_ms = (time.perf_counter() - start) / len(paths) * 1000
y_pred = probs.argmax(axis=1)
confidence = probs.max(axis=1)
print(f"batched prediction: {per_image_ms:.1f} ms per image on this Colab machine")

## 2. Accuracy, per-class precision / recall / F1

**Why accuracy alone is not enough:** it hides weak classes. A model could be 95 % accurate overall
but confuse two blights most of the time. Per-class **precision** (when it says X, how often is it X?)
and **recall** (of all real X, how many did it find?) show this. **F1** combines both.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def pretty(raw):
    crop, _, cond = raw.partition("___")
    crop = {"Pepper,_bell": "Pepper", "Corn": "Corn"}.get(crop, crop)
    cond = cond.replace("_", " ").replace("Tomato ", "")
    return f"{crop}: {cond}"

short = [pretty(l) for l in labels]
accuracy = accuracy_score(y_true, y_pred)
report_text = classification_report(y_true, y_pred, target_names=short, digits=3)
report = classification_report(y_true, y_pred, target_names=short, output_dict=True)
(OUT_DIR / "reports" / "classification_report.txt").write_text(report_text)
print(f"Test accuracy: {accuracy:.4f}\n")
print(report_text)

weak = [(name, report[name]["f1-score"]) for name in short if report[name]["f1-score"] < 0.80]
print("Classes with F1 < 0.80:", weak or "none")

## 3. Confusion matrix

Row = true class, column = predicted class. The diagonal is correct; anything off the diagonal
is a mistake. We list the two biggest confusions so we can explain them.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
fig, ax = plt.subplots(figsize=(12, 11))
ConfusionMatrixDisplay(cm, display_labels=short).plot(ax=ax, cmap="Greens", colorbar=False, xticks_rotation=90)
ax.set_title(f"LeafCheck test set confusion matrix (accuracy {accuracy:.1%})")
plt.tight_layout()
plt.savefig(OUT_DIR / "reports" / "confusion_matrix.png", dpi=120)
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
top_confusions = []
for flat in np.argsort(off, axis=None)[::-1][:2]:
    t, p = np.unravel_index(flat, off.shape)
    if off[t, p] > 0:
        top_confusions.append((short[t], short[p], int(off[t, p])))
for t, p, n in top_confusions:
    print(f"{n} x  true '{t}'  predicted as '{p}'")

## 4. The "Not sure" threshold

The app refuses to name a class when the top confidence is below 0.60. Here we measure how often
that happens on the test set, and how accurate the model is when it **is** confident.

In [ ]:
confident = confidence >= THRESHOLD
not_sure_rate = 1 - confident.mean()
acc_confident = (y_pred[confident] == y_true[confident]).mean() if confident.any() else float("nan")
acc_unsure = (y_pred[~confident] == y_true[~confident]).mean() if (~confident).any() else float("nan")
print(f"'Not sure' rate:                 {not_sure_rate:.2%}")
print(f"Accuracy on confident predictions: {acc_confident:.2%}")
print(f"Accuracy on 'not sure' ones:       {acc_unsure:.2%}")

## 5. Write `reports/results.md`

In [ ]:
history = json.loads((OUT_DIR / "reports" / "history.json").read_text())
rows = "\n".join(
    f"| {name} | {report[name]['precision']:.3f} | {report[name]['recall']:.3f} | "
    f"{report[name]['f1-score']:.3f} | {int(report[name]['support'])} |{' ⚠' if report[name]['f1-score'] < 0.80 else ''}"
    for name in short)
confusions = "\n".join(f"{i+1}. **{t}** predicted as **{p}** ({n} images)"
                        for i, (t, p, n) in enumerate(top_confusions)) or "None."

results_md = f"""# LeafCheck evaluation results

Held-out **test set**: {len(filenames)} images (15 % stratified split, seed 42), never used in training.

| Metric | Value | Target |
| --- | --- | --- |
| Test accuracy | **{accuracy:.2%}** | >= 90 % |
| Macro F1 | {report['macro avg']['f1-score']:.3f} | |
| Classes with F1 < 0.80 | {len(weak)} | flag any |
| "Not sure" rate (confidence < {THRESHOLD}) | {not_sure_rate:.2%} | report |
| Accuracy on confident predictions | {acc_confident:.2%} | report |
| Epochs trained (early stopping) | {len(history['loss'])} | ~10 |
| Best validation accuracy | {max(history['val_accuracy']):.2%} | |
| Prediction time, cache miss (laptop CPU) | _fill in from /stats_ | < 1 s |
| Prediction time, cache hit | _fill in from /stats_ | a few ms |

## Per-class results

| Class | Precision | Recall | F1 | Test images |
| --- | --- | --- | --- | --- |
{rows}

## Top confusions

{confusions}

![Confusion matrix](confusion_matrix.png)

![Training curves](training_curves.png)

## Caveat

PlantVillage photos are single leaves on plain backgrounds taken in a lab. Accuracy on real
field photos (mixed backgrounds, several leaves, different lighting) will be lower.
"""
(OUT_DIR / "reports" / "results.md").write_text(results_md)
print(results_md)

## 6. Export demo images and the sanity-test sample

* `samples/test_split/`: 2 test images per class + `expected.json` → used by `tests/test_model_sanity.py`.
* `samples/*.jpg`: a few nicely named demo images for the live demo.

**Add yourself afterwards:** 2 phone photos of real leaves and 1 non-leaf photo (e.g. a cup)
into `samples/` for the "Not sure" demo.

In [ ]:
sample_dir = OUT_DIR / "samples"
sanity_dir = sample_dir / "test_split"
sanity_dir.mkdir(parents=True, exist_ok=True)
sanity_expected, per_class = {}, {}
for f in filenames:
    label = expected[f]
    if per_class.get(label, 0) < 2:
        shutil.copy(test_dir / f, sanity_dir / f)
        sanity_expected[f] = label
        per_class[label] = per_class.get(label, 0) + 1
(sanity_dir / "expected.json").write_text(json.dumps(sanity_expected, indent=1))

demo = {"Tomato___Early_blight": "tomato_early_blight.jpg", "Tomato___healthy": "tomato_healthy.jpg",
        "Potato___healthy": "potato_healthy.jpg", "Potato___Late_blight": "potato_late_blight.jpg",
        "Corn___Common_rust": "corn_common_rust.jpg", "Pepper,_bell___Bacterial_spot": "pepper_bacterial_spot.jpg"}
for f, i in zip(filenames, range(len(filenames))):
    label = expected[f]
    # pick a demo image the model got right with high confidence
    if label in demo and y_pred[i] == y_true[i] and confidence[i] > 0.9:
        shutil.copy(test_dir / f, sample_dir / demo.pop(label))
print("sanity images:", len(sanity_expected), "| demo images left unfilled:", list(demo))

## 7. Download everything as one zip → unzip into the repo root

In [ ]:
import os
from google.colab import files

os.chdir(OUT_DIR)
!rm -f /content/leafcheck_outputs.zip
!zip -qr /content/leafcheck_outputs.zip model reports samples
files.download("/content/leafcheck_outputs.zip")